In [1]:
from pynq import Overlay, MMIO
import pynq.lib as lib
from pynq import allocate
import numpy as np
from pynq.lib import DMA
import time

ol = Overlay("/root/jupyter_notebooks/rhd_spi_0807/AXIS_v1/rhd_spi_ip_0806_wrapper.bit")

In [2]:
ol.ip_dict.keys()

dict_keys(['axi_rhd_spi_master_0', 'axi_dma_0', 'zynq_ultra_ps_e_0'])

In [3]:
dma = ol.axi_dma_0
dma.register_map.MM2S_DMACR

Register(RS=0, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0)

In [4]:
dma.register_map

RegisterMap {
  MM2S_DMACR = Register(RS=0, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  MM2S_DMASR = Register(Halted=1, Idle=0, SGIncld=1, DMAIntErr=0, DMASlvErr=0, DMADecErr=0, SGIntErr=0, SGSlvErr=0, SGDecErr=0, IOC_Irq=0, Dly_Irq=0, Err_Irq=0, IRQThresholdSts=1, IRQDelaySts=0),
  MM2S_CURDESC = Register(Current_Descriptor_Pointer=0),
  MM2S_CURDESC_MSB = Register(Current_Descriptor_Pointer=0),
  MM2S_TAILDESC = Register(Tail_Descriptor_Pointer=0),
  MM2S_TAILDESC_MSB = Register(Tail_Descriptor_Pointer=0),
  MM2S_SA = Register(Source_Address=0),
  MM2S_SA_MSB = Register(Source_Address=0),
  MM2S_LENGTH = Register(Length=0),
  SG_CTL = Register(SG_CACHE=3, SG_USER=0),
  S2MM_DMACR = Register(RS=0, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  S2MM_DMASR = Register(Halted=1, Idle=0, SGIncld=1, DMAIntErr=0, DMASlvErr=0, DMADecErr=0, SGIntErr=0, SGSlvErr=0, SG

In [5]:
# 分配緩存區
data_size = 35
RX_buffer = allocate(shape = (data_size,), dtype = np.uint16)

print("RX Buf Addr:", hex(RX_buffer.physical_address))
print("---")
print("DMA Source address     :", hex(dma.register_map.MM2S_SA.Source_Address))
print("DMA Destination address:", hex(dma.register_map.S2MM_DA.Destination_Address))

RX Buf Addr: 0x2ad9000
---
DMA Source address     : 0x0
DMA Destination address: 0x0


In [6]:
# SPI setting
# RHD SPI IP 的基地址 
RHD_SPI_BASE = 0xA0000000
ADDRESS_RANGE = 0x10000

# 暫存器偏移
CONTROL_REG = 0x00  # bit[0]=start_pulse, bit[1]=stop_pulse, bit[7:4]=phase_select
PHA_SEL_REG = 0x04  # [3:0]
TX_DATA_REG = 0x08  # 16-bit data to transmit
RX_DATA_REG = 0x0C  # 16-bit received data
STATUS_REG  = 0x10  # bit[0]=busy, bit[1]=finish
IE_IS_REG   = 0x14  # 中斷控制暫存器
DATA_RO_REG = 0x18  # data valid flag
# 初始化 MMIO
mmio = MMIO(RHD_SPI_BASE, ADDRESS_RANGE)

# 讀取狀態
status = mmio.read(STATUS_REG)
print(f"SPI Status: 0x{status:08x}")
print(f"Busy: {status & 0x1}")
print(f"Finish: {(status >> 1) & 0x1}")

SPI Status: 0x00000001
Busy: 1
Finish: 0


In [191]:
# open SPI transfer
mmio.write(PHA_SEL_REG, 0x1)
mmio.write(CONTROL_REG, 0x1)
time.sleep(0.1)
mmio.write(CONTROL_REG, 0x2)

In [176]:
# open DMA transfer

dma.recvchannel.transfer(RX_buffer)
dma.recvchannel.wait()
for i in range(35):
    print(f"data {i:2d}: 0x{int(RX_buffer[i]):04X}")


KeyboardInterrupt: 

In [192]:
N = 55  # 跑幾次

# # 可選：先預熱一次，避免第一次呼叫偏慢影響平均
# dma.recvchannel.transfer(RX_buffer)
# dma.recvchannel.wait()

t0 = time.perf_counter()
for _ in range(N):
    dma.recvchannel.transfer(RX_buffer)
    dma.recvchannel.wait()
#     for i in range(35):
#         print(f"data {i:2d}: 0x{int(RX_buffer[i]):04X}")
    print(_)
t1 = time.perf_counter()

elapsed = t1 - t0
per_iter = elapsed / N
fps = 1.0 / per_iter
throughput_mib_s = (RX_buffer.nbytes / per_iter) / (1024 * 1024)

print(f"iters={N}, elapsed={elapsed:.3f}s")
print(f"per_iter={per_iter*1e3:.3f} ms  |  rate={fps:.1f} it/s")
print(f"payload={RX_buffer.nbytes} bytes/iter  |  throughput≈{throughput_mib_s:.2f} MiB/s")


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
iters=55, elapsed=0.063s
per_iter=1.152 ms  |  rate=868.2 it/s
payload=70 bytes/iter  |  throughput≈0.06 MiB/s


In [193]:
import time
import numpy as np

# ==== 你的既有配置（假設已經做過）====
# data_size = 100
# RX_buffer = allocate(shape=(data_size,), dtype=np.uint16)

FRAME_SAMPLES = 35
BTT = FRAME_SAMPLES * 2     # 70 bytes
CAPTURE_S = 0.10            # 固定擷取時間（秒）→ 你原本 sleep(0.1)

# ---- 先 arm 一個接收，讓 DMA "在等資料" ----
dma.recvchannel.transfer(RX_buffer, nbytes=BTT)

# ---- 開 SPI 並啟動上游 ----
mmio.write(PHA_SEL_REG, 0x1)
mmio.write(CONTROL_REG, 0x1)

# ---- 時間窗開始 ----
t0 = time.perf_counter()
t_end = t0 + CAPTURE_S

frames = []   # 每個元素是本次 frame 的 35 筆拷貝

while True:
    # 等一個 frame 完成（由上游 TLAST 結束）
    dma.recvchannel.wait()
    frames.append(RX_buffer[:FRAME_SAMPLES].copy())

    # 到時限就跳出；否則立刻 arm 下一個 frame
    if time.perf_counter() >= t_end:
        break
    dma.recvchannel.transfer(RX_buffer, nbytes=BTT)

# ---- 到這裡不再 arm 新的 transfer；關閉上游 ----
mmio.write(CONTROL_REG, 0x2)

t1 = time.perf_counter()

# ==== 簡單統計 ====
num_frames = len(frames)
elapsed = t1 - t0
rate = num_frames / elapsed if elapsed > 0 else 0.0
throughput_mib_s = (num_frames * BTT) / (elapsed * 1024 * 1024) if elapsed > 0 else 0.0

print(f"captured {num_frames} frames in {elapsed:.3f}s "
      f"→ {rate:.1f} fps, ~{throughput_mib_s:.3f} MiB/s")

# （可選）檢查前 1 個 frame 的前 10 筆
if num_frames:
    print("first frame head:", [f"0x{int(v):04X}" for v in frames[0][:10]])


captured 89 frames in 0.101s → 877.1 fps, ~0.059 MiB/s
first frame head: ['0xEB00', '0xEC00', '0xE800', '0xE900', '0xEA00', '0xEB00', '0xEC00', '0xE800', '0xE900', '0xEA00']


In [202]:
import numpy as np, time
from pynq import allocate

FRAME_BYTES = 35 * 2          # 70 bytes per frame
K = 2048                      # BD 數量（環大小；可依需求調）
BD_BYTES = 0x40               # 每個 RX BD 長度固定 64B（AXI DMA 規格）
assert BD_BYTES % 64 == 0

# 資料環與 BD 環（關掉 cache 省去 invalidate 麻煩）
RX_ring = allocate(shape=(K * FRAME_BYTES,), dtype=np.uint8,  cacheable=False)
descs   = allocate(shape=(K * (BD_BYTES // 4),), dtype=np.uint32, cacheable=False)

def bd_phys(i):  return descs.physical_address   + i * BD_BYTES
def buf_phys(i): return RX_ring.physical_address + i * FRAME_BYTES

# 依 PG021 的 RX 描述子版面填入欄位：
#   0x00 NXTDESC          (下一個 BD 位址；最後一個指回 0，形成環)
#   0x04 NXTDESC_MSB      (Zynq 可為 0)
#   0x08 BUFFER_ADDRESS   (資料位址)
#   0x0C BUFFER_ADDRESS_MSB
#   0x10 CACHE/USER/CTL   (設 AWCACHE=0b0011；建議 0x03000000)
#   0x14 STRIDE_VSIZE     (VSIZE=1<<19, STRIDE=FRAME_BYTES)
#   0x18 HSIZE            (每列 bytes=FRAME_BYTES=70)
#   0x1C STATUS           (啟動前清 0；DMA 完成後置位 Cmp 等)
for i in range(K):
    base = i * (BD_BYTES // 4)
    descs[base + 0] = bd_phys((i + 1) % K)  # NXTDESC
    descs[base + 1] = 0                     # NXTDESC_MSB
    descs[base + 2] = buf_phys(i)           # BUFFER_ADDRESS
    descs[base + 3] = 0                     # BUFFER_ADDRESS_MSB
    descs[base + 4] = 0x03000000            # CACHE/USER/CTL
    descs[base + 5] = (1 << 19) | (FRAME_BYTES & 0xFFFF)  # VSIZE=1, STRIDE=70
    descs[base + 6] = FRAME_BYTES           # HSIZE=70
    descs[base + 7] = 0                     # STATUS 清 0

descs.flush()
RX_ring.flush()



In [203]:
rm = dma.register_map  # ★ 重點：改用 register_map

# 乾淨啟動：Reset（可選）
rm.S2MM_DMACR.Reset = 1
time.sleep(0.001)

# 指定第一個 BD
rm.S2MM_CURDESC = bd_phys(0)
rm.S2MM_CURDESC_MSB = 0

# 開 cyclic + RS=1
rm.S2MM_DMACR.Cyclic_BD_Enable = 1
rm.S2MM_DMACR.RS = 1

# kick：寫 tail（cyclic 下 tail 主要用於觸發；常見做法指向最後一個 BD）
rm.S2MM_TAILDESC = bd_phys(K - 1)
rm.S2MM_TAILDESC_MSB = 0


In [204]:
# 開來源
mmio.write(PHA_SEL_REG, 0x1)
mmio.write(CONTROL_REG, 0x1)

CAPTURE_S = 1.0
time.sleep(CAPTURE_S)

# 關來源
mmio.write(CONTROL_REG, 0x2)
time.sleep(0.001)

# 停 DMA（退出 cyclic）
rm.S2MM_DMACR.RS = 0

# 等 Halted=1
t0 = time.time()
while not rm.S2MM_DMASR.Halted and time.time() - t0 < 1.0:
    pass

# 若一開始把 buffer/BD 設為 cacheable=True，這裡要 invalidate()
RX_ring.invalidate()
descs.invalidate()


In [205]:
# 計算完成的 BD（frame）
BDW = BD_BYTES // 4
done = 0
for i in range(K):
    if (descs[i * BDW + 7] >> 31) & 1:  # STATUS[31] = Cmp
        done += 1
print("frames captured:", done)

# 取出前 done 個 frame（每幀 35×uint16）
total_samples = done * 35
cap = np.frombuffer(RX_ring, dtype=np.uint16, count=total_samples, offset=0)
cap = cap.reshape(done, 35)

print("first frame head:", [f"0x{int(v):04X}" for v in cap[0, :10]] if done else [])


frames captured: 2048
first frame head: ['0xEC00', '0xE800', '0xE900', '0xEA00', '0xEB00', '0xEC00', '0xE800', '0xE900', '0xEA00', '0xEB00']


In [8]:
# V1 (SG cyclic + moving tail)

import time
import numpy as np
from pynq import allocate

# ===== 參數（可依需求調整）====================================================
FRAME_SAMPLES   = 35                      # 每 frame 有 35 筆 uint16
FRAME_BYTES     = FRAME_SAMPLES * 2       # 70 bytes
BD_BYTES        = 0x40                    # AXI DMA RX BD 固定 64B
K               = 8192                    # 環上 BD/frames 數（可拉更大）
CAPTURE_S       = 4.0                     # 擷取時間（秒）
POLL_S          = 0.0005                  # 掃 BD Status 的週期（秒），0.5 ms 就夠

# ===== 1) 建「資料環」＋「BD 環」（cacheable=False 省掉 invalidate）==========
RX_ring = allocate(shape=(K * FRAME_BYTES,), dtype=np.uint8,  cacheable=False)
descs   = allocate(shape=(K * (BD_BYTES // 4),), dtype=np.uint32, cacheable=False)

# AXI DMA (PG021) 的 RX BD 佈局（以 32-bit words 計）：
#  0: NXTDESC (L)            @ 0x00
#  1: NXTDESC_MSB            @ 0x04
#  2: BUFFER_ADDRESS (L)     @ 0x08
#  3: BUFFER_ADDRESS_MSB     @ 0x0C
#  4: APP0/RSVD              @ 0x10  (不用)
#  5: APP1/RSVD              @ 0x14  (不用)
#  6: CONTROL/LENGTH         @ 0x18  ★ 低位元寫「本 BD 要收的 bytes」= 70
#  7: STATUS                 @ 0x1C  ★ 完成後硬體會把 bit31 = 1
#  8..15: APP2..APP5/RSVD    @ 0x20..0x3C（不用）
OFF_NXTDESC   = 0
OFF_NXTDESC_H = 1
OFF_BUF      = 2
OFF_BUF_H    = 3
OFF_CTRL     = 6
OFF_STS      = 7
BDW          = BD_BYTES // 4

def bd_phys(i):  return descs.physical_address   + i * BD_BYTES
def buf_phys(i): return RX_ring.physical_address + i * FRAME_BYTES

# 依 AXI DMA RX BD 格式填欄位（每個 BD 長度=70B）
for i in range(K):
    base = i * BDW
    descs[base + OFF_NXTDESC]   = bd_phys((i + 1) % K)   # 下一個 BD（最後一個指回 0）
    descs[base + OFF_NXTDESC_H] = 0
    descs[base + OFF_BUF]       = buf_phys(i)            # 此 BD 對應的資料起點
    descs[base + OFF_BUF_H]     = 0
    descs[base + OFF_CTRL]      = FRAME_BYTES            # ★ 長度（bytes）
    descs[base + OFF_STS]       = 0                      # ★ Status 清 0（完成後硬體會寫 bit31=1）

descs.flush()
RX_ring.flush()

# ===== 2) 啟動 S2MM（SG 模式，非 cyclic；用 moving tail 讓它無限前進）=========
rm = dma.register_map

# Reset（乾淨啟動）
rm.S2MM_DMACR.Reset = 1
time.sleep(0.001)

# 設定起點 BD
rm.S2MM_CURDESC     = bd_phys(0)
rm.S2MM_CURDESC_MSB = 0

# 關閉 cyclic（我們要自己補尾）
rm.S2MM_DMACR.Cyclic_BD_Enable = 0

# Run（RS=1）
rm.S2MM_DMACR.RS = 1

# 先把整個環送進去（Tail = K-1）
rm.S2MM_TAILDESC     = bd_phys(K - 1)
rm.S2MM_TAILDESC_MSB = 0

# ===== 3) 固定時間「邊收邊取」：掃 BD Status.bit31（Complete）=================
u16     = RX_ring.view(np.uint16)
head    = 0                    # 下一個等待被「消費」的 BD 索引
tail    = (K - 1)             # 目前硬體「允許前進的最後一個」BD（我們會不斷往前推）
chunks  = []

# 開資料來源
mmio.write(PHA_SEL_REG, 0x1)
mmio.write(CONTROL_REG, 0x1)

t0 = time.perf_counter()
while time.perf_counter() - t0 < CAPTURE_S:
    # （非必要，但若你的 descs 設 cacheable=True，這裡要 descs.invalidate()）
    ready = 0
    # 連續地從 head 開始，數有多少個 BD 已完成（Status bit31=1）
    while ready < K:
        base = ( (head + ready) % K ) * BDW
        sts  = int(descs[base + OFF_STS])
        if (sts >> 31) & 1:          # 完成
            ready += 1
        else:
            break

    if ready:
        # 批次把 [head .. head+ready-1] 這段搬出來
        start_s = head * FRAME_SAMPLES
        if head + ready <= K:
            block = u16[start_s : start_s + ready*FRAME_SAMPLES] \
                        .reshape(ready, FRAME_SAMPLES).copy()
        else:
            n1 = K - head
            part1 = u16[start_s : K*FRAME_SAMPLES]
            n2 = ready - n1
            part2 = u16[0 : n2*FRAME_SAMPLES]
            block = np.concatenate([part1, part2]) \
                        .reshape(ready, FRAME_SAMPLES).copy()
        chunks.append(block)

        # 把這些 BD 的 Status 清 0，表示「已消費、可重用」
        for j in range(ready):
            base = ( (head + j) % K ) * BDW
            descs[base + OFF_STS] = 0
        # 如果 descs 是 cacheable=True，要 flush；此處我們是 False 可省略
        # descs.flush()

        # 推動 head
        head = (head + ready) % K

        # ★ 進一步「補尾」：把 TAILDESC 往前推 ready 個，讓 DMA 繼續無限前進
        tail = (tail + ready) % K
        rm.S2MM_TAILDESC     = bd_phys(tail)
        rm.S2MM_TAILDESC_MSB = 0

    time.sleep(POLL_S)         # 低頻掃描即可（0.5 ms）

# 關來源
mmio.write(CONTROL_REG, 0x2)
t_stop = time.perf_counter()

# 停 DMA
time.sleep(0.001)
rm.S2MM_DMACR.RS = 0

# 最多等 1 秒直到 Halted
t_wait = time.perf_counter()
while not rm.S2MM_DMASR.Halted and time.perf_counter() - t_wait < 1.0:
    pass

# ===== 4) 彙整結果 ==========================================================
cap = np.vstack(chunks) if chunks else np.empty((0, FRAME_SAMPLES), np.uint16)

T_eff = t_stop - t0
fps   = cap.shape[0] / T_eff if T_eff > 0 else 0.0
thr   = (cap.shape[0] * FRAME_BYTES) / (T_eff * 1024 * 1024) if T_eff > 0 else 0.0

print(f"live captured: {cap.shape[0]} frames in {T_eff:.6f}s  "
      f"({fps:.1f} fps; ~{thr:.3f} MiB/s)")
if cap.shape[0]:
    print("first frame head:", [f"0x{int(v):04X}" for v in cap[0, :10]])

# ===== 5) 存檔（每幀一列、35 欄）===========================================
if cap.size:
    stamp = time.strftime("%Y%m%d_%H%M%S")
    base  = f"/home/xilinx/cap_{cap.shape[0]}f_{stamp}"

    # 十六進位（每格像 0xE800），用逗號分隔，讀起來清楚
    np.savetxt(base + "_hex.txt", cap, fmt="0x%04X", delimiter=",")

    # 十進位版本（若你後處理要數值型）
    np.savetxt(base + "_dec.txt", cap, fmt="%d", delimiter=",")

    print(f"saved: {base}_hex.txt  and  {base}_dec.txt")
else:
    print("cap is empty; skip saving.")

live captured: 142855 frames in 8.001305s  (17854.0 fps; ~1.192 MiB/s)
first frame head: ['0xFF00', '0xFF00', '0x80DE', '0x8142', '0x8204', '0x8300', '0x8480', '0x8540', '0x8680', '0x8700']
saved: /home/xilinx/cap_142855f_20250903_160925_hex.txt  and  /home/xilinx/cap_142855f_20250903_160925_dec.txt


In [7]:
# V3 (SG + moving tail, STREAM, high-throughput)
# 目標：錄製期間只寫二進位 .bin，避免文字 I/O 拖慢採樣；事後再轉文字
# 頻率改變時，PHA_SEL_REG也要改

import time
import numpy as np
from pathlib import Path
from pynq import allocate

# ===== 可調參數 ===============================================================
FRAME_SAMPLES   = 35
FRAME_BYTES     = FRAME_SAMPLES * 2       # 70 bytes
BD_BYTES        = 0x40                    # 64B
K               = 65536                   # 放大 BD/frames 環（建議 32768 或 65536）
CAPTURE_S       = 10.0                    # recode time
POLL_S          = 0.0                     # 盡量不 sleep

# 只開二進位（最快最小）；如要文字，錄完後離線轉
SAVE_BIN        = True
SAVE_HEX_TXT    = False                   # 不建議邊收邊寫文字；會降速
SAVE_DEC_TXT    = False

# 與程式同目錄
OUT_DIR         = Path.cwd()
STAMP           = time.strftime("%Y%m%d_%H%M%S")
BASE_NAME       = f"cap_stream_{STAMP}"

# ===== 建資料/BD 環（非快取，免 invalidate）===================================
RX_ring = allocate(shape=(K * FRAME_BYTES,), dtype=np.uint8,  cacheable=False)
descs   = allocate(shape=(K * (BD_BYTES // 4),), dtype=np.uint32, cacheable=False)

OFF_NXTDESC   = 0
OFF_NXTDESC_H = 1
OFF_BUF      = 2
OFF_BUF_H    = 3
OFF_CTRL     = 6
OFF_STS      = 7
BDW          = BD_BYTES // 4

def bd_phys(i):  return descs.physical_address   + i * BD_BYTES
def buf_phys(i): return RX_ring.physical_address + i * FRAME_BYTES

for i in range(K):
    base = i * BDW
    descs[base + OFF_NXTDESC]   = bd_phys((i + 1) % K)
    descs[base + OFF_NXTDESC_H] = 0
    descs[base + OFF_BUF]       = buf_phys(i)
    descs[base + OFF_BUF_H]     = 0
    descs[base + OFF_CTRL]      = FRAME_BYTES
    descs[base + OFF_STS]       = 0

descs.flush()
RX_ring.flush()

# ===== 啟動 S2MM（SG + moving tail）===========================================
rm = dma.register_map  # 你現有的 dma 物件

rm.S2MM_DMACR.Reset = 1
time.sleep(0.001)

rm.S2MM_CURDESC     = bd_phys(0)
rm.S2MM_CURDESC_MSB = 0
rm.S2MM_DMACR.Cyclic_BD_Enable = 0   # 保持 SG + moving tail
rm.S2MM_DMACR.RS = 1
rm.S2MM_TAILDESC     = bd_phys(K - 1)
rm.S2MM_TAILDESC_MSB = 0

# ===== 開檔（只二進位）========================================================
OUT_DIR.mkdir(parents=True, exist_ok=True)
f_bin = open(OUT_DIR / f"{BASE_NAME}.bin", "wb") if SAVE_BIN else None
# 如你硬要邊收邊寫文字，打開下兩行，但吞吐會掉：
f_hex = open(OUT_DIR / f"{BASE_NAME}_hex.txt", "w", buffering=1024*1024) if SAVE_HEX_TXT else None
f_dec = open(OUT_DIR / f"{BASE_NAME}_dec.txt", "w", buffering=1024*1024) if SAVE_DEC_TXT else None

# ===== 邊收邊寫（無拷貝；wrap 分兩段直寫）====================================
u16   = RX_ring.view(np.uint16)
head  = 0
tail  = K - 1
nfrm  = 0

# 開資料來源（依你的設計）
mmio.write(PHA_SEL_REG, 0x2)  ## 25M max at 0x03
mmio.write(CONTROL_REG, 0x1)

t0 = time.perf_counter()
try:
    while time.perf_counter() - t0 < CAPTURE_S:
        # 計數連續完成的 BD
        ready = 0
        while ready < K:
            base = ((head + ready) % K) * BDW
            sts  = int(descs[base + OFF_STS])
            if (sts >> 31) & 1:
                ready += 1
            else:
                break

        if ready:
            # 連續區段（不 wrap）
            start = head * FRAME_SAMPLES
            total_samp = ready * FRAME_SAMPLES

            if head + ready <= K:
                view1d = u16[start : start + total_samp]  # 1D view，無 copy
                # 二進位最快：直接把 view1d 以 <u2 寫出
                if f_bin is not None:
                    view1d.astype('<u2', copy=False).tofile(f_bin)

                # （不建議）文字：一列 35 欄
                if f_hex is not None or f_dec is not None:
                    view2d = view1d.reshape(ready, FRAME_SAMPLES)
                    if f_hex is not None:
                        for row in view2d:
                            f_hex.write(",".join(f"0x{int(x):04X}" for x in row) + "\n")
                    if f_dec is not None:
                        for row in view2d:
                            f_dec.write(",".join(str(int(x)) for x in row) + "\n")

            else:
                # wrap：分兩段寫，避免 concat/copy
                n1 = K - head
                part1_1d = u16[start : K*FRAME_SAMPLES]
                n2 = ready - n1
                part2_1d = u16[0 : n2*FRAME_SAMPLES]

                if f_bin is not None:
                    part1_1d.astype('<u2', copy=False).tofile(f_bin)
                    part2_1d.astype('<u2', copy=False).tofile(f_bin)

                if f_hex is not None or f_dec is not None:
                    part1_2d = part1_1d.reshape(n1, FRAME_SAMPLES)
                    part2_2d = part2_1d.reshape(n2, FRAME_SAMPLES)
                    if f_hex is not None:
                        for row in part1_2d:
                            f_hex.write(",".join(f"0x{int(x):04X}" for x in row) + "\n")
                        for row in part2_2d:
                            f_hex.write(",".join(f"0x{int(x):04X}" for x in row) + "\n")
                    if f_dec is not None:
                        for row in part1_2d:
                            f_dec.write(",".join(str(int(x)) for x in row) + "\n")
                        for row in part2_2d:
                            f_dec.write(",".join(str(int(x)) for x in row) + "\n")

            nfrm += ready

            # 清 status、推 head/tail（無 flush，因為 non-cacheable）
            for j in range(ready):
                base = ((head + j) % K) * BDW
                descs[base + OFF_STS] = 0

            head = (head + ready) % K
            tail = (tail + ready) % K
            rm.S2MM_TAILDESC     = bd_phys(tail)
            rm.S2MM_TAILDESC_MSB = 0

        if POLL_S:
            time.sleep(POLL_S)

finally:
    # 關來源 & 停 DMA
    mmio.write(CONTROL_REG, 0x2)
    t_stop = time.perf_counter()

    time.sleep(0.001)
    rm.S2MM_DMACR.RS = 0
    t_wait = time.perf_counter()
    while not rm.S2MM_DMASR.Halted and time.perf_counter() - t_wait < 1.0:
        pass

    # 關檔
    if f_bin: f_bin.close()
    if f_hex: f_hex.close()
    if f_dec: f_dec.close()

# ===== 統計 ===================================================================
T_eff = t_stop - t0
fps   = nfrm / T_eff if T_eff > 0 else 0.0
thr   = (nfrm * FRAME_BYTES) / (T_eff * 1024 * 1024) if T_eff > 0 else 0.0
print(f"[STREAM] captured: {nfrm} frames in {T_eff:.6f}s  ({fps:.1f} fps; ~{thr:.3f} MiB/s)")
print("saved to:", str(OUT_DIR / f"{BASE_NAME}.bin"))


[STREAM] captured: 300001 frames in 10.000248s  (29999.4 fps; ~2.003 MiB/s)
saved to: /root/jupyter_notebooks/rhd_spi_0807/AXIS_v1/cap_stream_20250929_233831.bin


In [9]:
# 轉檔
import numpy as np

def bin_to_txt(bin_path, frame_samples=35, out_hex=None, out_dec=None):
    arr = np.fromfile(bin_path, dtype="<u2").reshape(-1, frame_samples)
    if out_hex:
        np.savetxt(out_hex, arr, fmt="0x%04X", delimiter=",")
#     if out_dec:
#         np.savetxt(out_dec, arr, fmt="%d", delimiter=",")

# 用法：
# bin_to_txt(f"{BASE_NAME}.bin", out_hex=f"{BASE_NAME}_hex.txt", out_dec=f"{BASE_NAME}_dec.txt")
bin_to_txt(f"{BASE_NAME}.bin", out_hex=f"{BASE_NAME}_hex.txt")
print(f"{BASE_NAME} Finish")


cap_stream_20250930_002120 Finish


In [19]:
mmio.write(CONTROL_REG, 0x1)

In [37]:
# SG 非 cyclic，固定 N 個 BD

import time
import numpy as np
from pynq import allocate

# ====== 你的目標 ======
FS_HZ      = 300000        # 每通道目標 sample rate（= frame rate）
CAPTURE_S  = 1.0          # 要抓幾秒

# ====== 衍生參數 ======
FRAME_SAMPLES = 35
FRAME_BYTES   = FRAME_SAMPLES * 2       # 70 bytes
BD_BYTES      = 0x40                    # AXI DMA RX BD 固定 64B
BDW           = BD_BYTES // 4

N_FRAMES      = int(round(FS_HZ * CAPTURE_S))
DATA_BYTES    = N_FRAMES * FRAME_BYTES
# print(f"plan: {N_FRAMES} frames, {DATA_BYTES} bytes")

# ====== 分配緩衝（cacheable=False 省掉 invalidate 煩惱）======
RX_buf = allocate(shape=(DATA_BYTES,), dtype=np.uint8,  cacheable=False)
descs  = allocate(shape=(N_FRAMES * BDW,), dtype=np.uint32, cacheable=False)

# BD 欄位 offset（AXI DMA PG021）
OFF_NXTDESC   = 0   # 0x00
OFF_NXTDESC_H = 1   # 0x04
OFF_BUF       = 2   # 0x08
OFF_BUF_H     = 3   # 0x0C
OFF_CTRL      = 6   # 0x18  ← 寫此次 BD 長度（bytes）
OFF_STS       = 7   # 0x1C  ← 完成後 bit31=1

def bd_phys(i):  return descs.physical_address   + i * BD_BYTES
def buf_phys(i): return RX_buf.physical_address  + i * FRAME_BYTES

# ====== 填 BD：一共 N_FRAMES 個，每個長度 70B，最後一個 NXTDESC=0 ======
for i in range(N_FRAMES):
    base = i * BDW
    descs[base + OFF_NXTDESC]   = bd_phys(i+1) if i < N_FRAMES-1 else 0
    descs[base + OFF_NXTDESC_H] = 0
    descs[base + OFF_BUF]       = buf_phys(i)
    descs[base + OFF_BUF_H]     = 0
    descs[base + OFF_CTRL]      = FRAME_BYTES
    descs[base + OFF_STS]       = 0

descs.flush(); RX_buf.flush()

# ====== 啟動 S2MM（SG，非 cyclic；尾指到第 N-1 個）======
rm = dma.register_map
rm.S2MM_DMACR.Reset = 1
time.sleep(0.001)

rm.S2MM_CURDESC     = bd_phys(0)
rm.S2MM_CURDESC_MSB = 0
rm.S2MM_DMACR.Cyclic_BD_Enable = 0    # ★ 關閉 cyclic（我們要固定幀數）
rm.S2MM_DMACR.RS = 1

rm.S2MM_TAILDESC     = bd_phys(N_FRAMES - 1)  # ★ 尾指向最後一個 BD
rm.S2MM_TAILDESC_MSB = 0

# ====== 開來源，開始取數 ======
mmio.write(PHA_SEL_REG, 0x2)
t0 = time.perf_counter()
mmio.write(CONTROL_REG, 0x1)

# 等「最後一個 BD 完成」（精準到幀）
last_base = (N_FRAMES - 1) * BDW
while True:
    sts = int(descs[last_base + OFF_STS])
    if (sts >> 31) & 1:                 # bit31 = Complete
        break
    # 也可監控錯誤：rm.S2MM_DMASR.DMAIntErr / DMASlvErr / DMADecErr ...\
    time.sleep(0.0005)

# 立刻關來源（避免多餘幀進來，但 DMA 已停在 tail，不會寫 DDR 了）
mmio.write(CONTROL_REG, 0x2)
t1 = time.perf_counter()

# 等 DMA Idle（保險）
t_wait = time.perf_counter()
while not rm.S2MM_DMASR.Idle and time.perf_counter() - t_wait < 1.0:
    pass
rm.S2MM_DMACR.RS = 0

# ====== 轉型成 (N_FRAMES, 35) 的 uint16 矩陣 ======
u16 = RX_buf.view(np.uint16).reshape(N_FRAMES, FRAME_SAMPLES)
print(f"captured {u16.shape[0]} frames in {t1 - t0:.6f}s")

# 看前幾個 word（十六進位）
print("first frame head:", [f"0x{int(v):04X}" for v in u16[0, :100]])


captured 300000 frames in 10.000953s
first frame head: ['0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0xFFFF', '0x0000', '0x0000', '0x0000', '0x0000']


In [35]:
# =============================================================================
# PYNQ AXI DMA S2MM TLAST 健康檢查 & 高速串流測試（30 kS/s 友善）
# - SG BD 環 + non-cacheable 資料環
# - 移動 tail（moving tail），持續收包
# - 邊收邊寫 .bin 到當前目錄
# - 週期性列印 S2MM 狀態與前幾個 BD 的 STATUS
# - 若在 HEALTHCHECK_S 內沒有任何 BD 完成 => 高機率 "TLAST 遺失"
# =============================================================================
# !!這個版本好像有點問題，使用後會無法控制 AXI-lite?

import time
import numpy as np
from pathlib import Path
from pynq import allocate

# ===== 你可以調整的參數 =======================================================
FRAME_SAMPLES   = 35           # 每 frame 的 16-bit words 數（= 35）
FRAME_BYTES     = FRAME_SAMPLES * 2
BD_BYTES        = 0x40         # AXI DMA RX BD 固定 64B
K               = 32768        # BD/frames 環大小（建議 32768 或 65536）
CAPTURE_S       = 2.0          # 總錄製秒數（可調大）
POLL_S          = 0.0          # 0=不 sleep；也可設 0.0005 之類
HEALTHCHECK_S   = 0.50         # 若這段時間內 "0 個 BD 完成" => 判定異常

# 如需啟停你的資料來源（沒有就註解）
PHA_SEL_REG = 0x00_000000      # ← 你的相位/模式控制暫存器位址（沒有就忽略）
CONTROL_REG = 0x00_000004      # ← 啟停暫存器位址（沒有就忽略）
SET_PHASE_VALUE = 0x1          # 你之前可用的 phase/mode 值（先用穩定那個）

SAVE_BIN = True                # 錄製時寫 .bin（建議開）
OUT_DIR  = Path.cwd()

# ======= 這兩個物件請先準備好（from overlays）===============================
# dma  = <你的 AXI DMA 物件>
# mmio = <你的控制暫存器 MMIO>  （沒有就把兩個 write 註解掉）

# ===== 工具函式 ===============================================================
def dump_s2mm_status(rm, descs, BD_BYTES, OFF_STS, head, tail):
    dmacr = int(rm.S2MM_DMACR)
    dmasr = int(rm.S2MM_DMASR)
    curd  = int(rm.S2MM_CURDESC)
    tdesc = int(rm.S2MM_TAILDESC)
    print(f"S2MM_DMACR=0x{dmacr:08X}  S2MM_DMASR=0x{dmasr:08X}  "
          f"CURDESC=0x{curd:08X}  TAILDESC=0x{tdesc:08X}  head={head} tail={tail}")
    print("  halted=", int(rm.S2MM_DMASR.Halted),
          " idle=",   int(rm.S2MM_DMASR.Idle),
          " IOC_Irq=",int(rm.S2MM_DMASR.IOC_Irq),
          " Err_Irq=",int(rm.S2MM_DMASR.Err_Irq))
    BDW = BD_BYTES // 4
    for i in range(4):
        base = i * BDW
        print(f"  BD{i} STS=0x{int(descs[base + OFF_STS]):08X}")

def build_rings(K, FRAME_BYTES, BD_BYTES):
    RX_ring = allocate(shape=(K * FRAME_BYTES,), dtype=np.uint8,  cacheable=False)
    descs   = allocate(shape=(K * (BD_BYTES // 4),), dtype=np.uint32, cacheable=False)

    OFF_NXTDESC   = 0
    OFF_NXTDESC_H = 1
    OFF_BUF       = 2
    OFF_BUF_H     = 3
    OFF_CTRL      = 6
    OFF_STS       = 7
    BDW           = BD_BYTES // 4

    def bd_phys(i):  return descs.physical_address   + i * BD_BYTES
    def buf_phys(i): return RX_ring.physical_address + i * FRAME_BYTES

    for i in range(K):
        base = i * BDW
        descs[base + OFF_NXTDESC]   = bd_phys((i + 1) % K)
        descs[base + OFF_NXTDESC_H] = 0
        descs[base + OFF_BUF]       = buf_phys(i)
        descs[base + OFF_BUF_H]     = 0
        descs[base + OFF_CTRL]      = FRAME_BYTES
        descs[base + OFF_STS]       = 0

    descs.flush(); RX_ring.flush()
    return RX_ring, descs, (OFF_NXTDESC, OFF_NXTDESC_H, OFF_BUF, OFF_BUF_H, OFF_CTRL, OFF_STS, BDW)

# ===== 主流程 =================================================================
STAMP     = time.strftime("%Y%m%d_%H%M%S")
BASE_NAME = f"cap_stream_{STAMP}"

RX_ring, descs, OFFS = build_rings(K, FRAME_BYTES, BD_BYTES)
OFF_NXTDESC, OFF_NXTDESC_H, OFF_BUF, OFF_BUF_H, OFF_CTRL, OFF_STS, BDW = OFFS

u16   = RX_ring.view(np.uint16)
head  = 0
tail  = K - 1
nfrm  = 0

# 檔案
OUT_DIR.mkdir(parents=True, exist_ok=True)
f_bin = open(OUT_DIR / f"{BASE_NAME}.bin", "wb") if SAVE_BIN else None

rm = dma.register_map

# 啟動前印一次狀態
print("=== BEFORE START ===")
dump_s2mm_status(rm, descs, BD_BYTES, OFF_STS, head, tail)

# 啟動資料來源（若沒有 mmio 控制，請註解）
try:
    mmio.write(PHA_SEL_REG, SET_PHASE_VALUE)
except Exception:
    pass
try:
    mmio.write(CONTROL_REG, 0x1)    # start
except Exception:
    pass

# 啟動 DMA（SG + moving tail）
rm.S2MM_DMACR.Reset = 1
time.sleep(0.001)
rm.S2MM_CURDESC     = descs.physical_address
rm.S2MM_CURDESC_MSB = 0
rm.S2MM_DMACR.Cyclic_BD_Enable = 0
rm.S2MM_DMACR.RS = 1
rm.S2MM_TAILDESC     = descs.physical_address + (K - 1) * BD_BYTES
rm.S2MM_TAILDESC_MSB = 0

print("=== AFTER START ===")
dump_s2mm_status(rm, descs, BD_BYTES, OFF_STS, head, tail)

t0 = time.perf_counter()
last_done_time = t0
last_print = t0
try:
    while time.perf_counter() - t0 < CAPTURE_S:
        ready = 0
        while ready < K:
            base = ((head + ready) % K) * BDW
            sts  = int(descs[base + OFF_STS])
            if (sts >> 31) & 1:      # 完成（代表這個 BD 看到了 TLAST）
                ready += 1
            else:
                break

        if ready:
            last_done_time = time.perf_counter()
            start = head * FRAME_SAMPLES
            total_samp = ready * FRAME_SAMPLES

            if head + ready <= K:
                view1d = u16[start : start + total_samp]
                if f_bin is not None:
                    view1d.astype('<u2', copy=False).tofile(f_bin)
            else:
                n1 = K - head
                part1_1d = u16[start : K*FRAME_SAMPLES]
                n2 = ready - n1
                part2_1d = u16[0 : n2*FRAME_SAMPLES]
                if f_bin is not None:
                    part1_1d.astype('<u2', copy=False).tofile(f_bin)
                    part2_1d.astype('<u2', copy=False).tofile(f_bin)

            nfrm += ready

            # 清 status、推 head/tail
            for j in range(ready):
                base = ((head + j) % K) * BDW
                descs[base + OFF_STS] = 0
            head = (head + ready) % K
            tail = (tail + ready) % K
            rm.S2MM_TAILDESC     = descs.physical_address + tail * BD_BYTES
            rm.S2MM_TAILDESC_MSB = 0

        # 週期性列印狀態 & 健康檢查
        now = time.perf_counter()
        if now - last_print >= 0.5:
            print(f"[{now - t0:6.3f}s] frames_done={nfrm}")
            dump_s2mm_status(rm, descs, BD_BYTES, OFF_STS, head, tail)
            last_print = now

        if (now - last_done_time) > HEALTHCHECK_S:
            print("!!! HealthCheck: 連續",
                  f"{HEALTHCHECK_S:.2f}s 都沒有 BD 完成 → 高機率 TLAST 遺失 / 分幀未到 / tkeep不一致")
            last_done_time = now  # 避免一直刷

        if POLL_S:
            time.sleep(POLL_S)

finally:
    # 停來源 & 停 DMA
    try:
        mmio.write(CONTROL_REG, 0x2)   # stop
    except Exception:
        pass

    t_stop = time.perf_counter()
    time.sleep(0.001)
    rm.S2MM_DMACR.RS = 0
    t_wait = time.perf_counter()
    while not rm.S2MM_DMASR.Halted and time.perf_counter() - t_wait < 1.0:
        pass

    if f_bin: f_bin.close()

# 總結
T_eff = t_stop - t0
fps   = nfrm / T_eff if T_eff > 0 else 0.0
thr   = (nfrm * FRAME_BYTES) / (T_eff * 1024 * 1024) if T_eff > 0 else 0.0
print(f"\n[STREAM] captured: {nfrm} frames in {T_eff:.6f}s  ({fps:.1f} fps; ~{thr:.3f} MiB/s)")
print("saved to:", str(OUT_DIR / f"{BASE_NAME}.bin"))
print("=== AFTER STOP ===")
dump_s2mm_status(rm, descs, BD_BYTES, OFF_STS, head, tail)



=== BEFORE START ===
S2MM_DMACR=0x00010002  S2MM_DMASR=0x00010008  CURDESC=0x35000040  TAILDESC=0x353FFFC0  head=0 tail=32767
  halted= 0  idle= 0  IOC_Irq= 0  Err_Irq= 0
  BD0 STS=0x00000000
  BD1 STS=0x00000000
  BD2 STS=0x00000000
  BD3 STS=0x00000000
=== AFTER START ===
S2MM_DMACR=0x00010003  S2MM_DMASR=0x00010008  CURDESC=0x35700040  TAILDESC=0x358FFFC0  head=0 tail=32767
  halted= 0  idle= 0  IOC_Irq= 0  Err_Irq= 0
  BD0 STS=0x00000000
  BD1 STS=0x00000000
  BD2 STS=0x00000000
  BD3 STS=0x00000000
[ 0.500s] frames_done=0
S2MM_DMACR=0x00010003  S2MM_DMASR=0x00010008  CURDESC=0x35700040  TAILDESC=0x358FFFC0  head=0 tail=32767
  halted= 0  idle= 0  IOC_Irq= 0  Err_Irq= 0
  BD0 STS=0x00000000
  BD1 STS=0x00000000
  BD2 STS=0x00000000
  BD3 STS=0x00000000
!!! HealthCheck: 連續 0.50s 都沒有 BD 完成 → 高機率 TLAST 遺失 / 分幀未到 / tkeep不一致
[ 1.000s] frames_done=0
S2MM_DMACR=0x00010003  S2MM_DMASR=0x00010008  CURDESC=0x35700040  TAILDESC=0x358FFFC0  head=0 tail=32767
  halted= 0  idle= 0  IOC_Irq= 0  E